# Lesson 2B: Dynamic Programming - Practical

## Introduction

In Lesson 2A, we learned the **theory** of Dynamic Programming: how policy iteration and value iteration work, and why they converge.

Now we go **deep into implementation**: efficiency tricks, advanced algorithms, and real-world optimization.

### Learning Objectives

1. Understand the **in-place vs. synchronous trade-off**
2. Implement **asynchronous DP** (updates in arbitrary order)
3. Implement **prioritized sweeping** (focus on high-impact states)
4. Build custom environments (GridWorld, Maze)
5. Benchmark algorithms and measure efficiency
6. Identify practical convergence issues and fixes

## Table of Contents

1. [Setup](#setup)
2. [Custom GridWorld Environment](#gridworld)
3. [In-Place vs. Synchronous Updates](#inplace)
4. [Asynchronous Dynamic Programming](#async)
5. [Prioritized Sweeping](#prioritized)
6. [Benchmarking & Comparison](#benchmarking)
7. [Debugging Convergence Issues](#debugging)
8. [Advanced: Generalized Policy Iteration](#gpi)
9. [Real-World Applications](#applications)

<a name="setup"></a>
## Setup

In [ ]:
import subprocess
import sys

packages = ['numpy', 'matplotlib', 'seaborn', 'pandas', 'scipy', 'gymnasium']
for pkg in packages:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import pandas as pd
from collections import defaultdict, deque
import time
from typing import Dict, List, Tuple, Set
import gymnasium as gym

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('✅ All libraries loaded!')

<a name="gridworld"></a>
## Custom GridWorld Environment

### Grid-Based MDP

A GridWorld is a simple MDP:
- **States**: Coordinates (i, j) on a grid
- **Actions**: Up, Down, Left, Right (4 discrete actions)
- **Rewards**: -1 per step, +10 at goal, -50 at obstacles
- **Transitions**: Deterministic (action succeeds) or stochastic (slip with prob p)

GridWorld is ideal for testing DP algorithms because:
- Fast to compute
- Easy to visualize
- Scalable (16, 64, 256 states)
- No external dependencies

In [ ]:
class GridWorld:
    """Simple GridWorld MDP for DP experiments."""
    
    def __init__(self, size=4, goal=(3, 3), obstacles=None, slip_prob=0.0):
        self.size = size
        self.goal = goal
        self.obstacles = set(obstacles) if obstacles else set()
        self.slip_prob = slip_prob
        
        # State and action spaces
        self.states = [(i, j) for i in range(size) for j in range(size) 
                      if (i, j) not in self.obstacles]
        self.actions = [0, 1, 2, 3]  # Up, Down, Left, Right
        self.action_names = ['↑', '↓', '←', '→']
        self.action_deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # (di, dj)
        
        self.state_to_idx = {s: i for i, s in enumerate(self.states)}
        self.idx_to_state = {i: s for i, s in enumerate(self.states)}
    
    def _is_valid(self, pos):
        """Check if position is valid and not an obstacle."""
        i, j = pos
        return 0 <= i < self.size and 0 <= j < self.size and pos not in self.obstacles
    
    def next_state(self, state, action):
        """Get next state and reward for (state, action)."""
        if state == self.goal:
            return state, 0  # Terminal state
        
        i, j = state
        di, dj = self.action_deltas[action]
        
        # Apply action with slip probability
        if np.random.random() < self.slip_prob:
            action = np.random.randint(4)  # Random action
            di, dj = self.action_deltas[action]
        
        next_pos = (i + di, j + dj)
        
        if not self._is_valid(next_pos):
            next_pos = state  # Stay in place
        
        # Reward: -1 per step, +10 at goal
        reward = 10.0 if next_pos == self.goal else -1.0
        
        return next_pos, reward
    
    def get_transitions(self, state, action):
        """Get deterministic transitions (no slip)."""
        if state == self.goal:
            return {state: 1.0}, {state: 0.0}
        
        i, j = state
        di, dj = self.action_deltas[action]
        next_pos = (i + di, j + dj)
        
        if not self._is_valid(next_pos):
            next_pos = state
        
        reward = 10.0 if next_pos == self.goal else -1.0
        
        return {next_pos: 1.0}, {next_pos: reward}
    
    def visualize_policy(self, policy, title='Policy'):
        """Visualize policy as arrows on grid."""
        fig, ax = plt.subplots(figsize=(6, 6))
        
        grid = np.zeros((self.size, self.size))
        for i in range(self.size):
            for j in range(self.size):
                if (i, j) in self.obstacles:
                    grid[i, j] = -1
        
        im = ax.imshow(grid, cmap='RdYlGn', vmin=-1, vmax=1, alpha=0.3)
        
        for state in self.states:
            i, j = state
            if state in policy:
                action = policy[state]
                ax.text(j, i, self.action_names[action], ha='center', va='center',
                       fontsize=14, fontweight='bold')
        
        # Mark goal
        gi, gj = self.goal
        ax.plot(gj, gi, 'r*', markersize=20, label='Goal')
        
        ax.set_xticks(range(self.size))
        ax.set_yticks(range(self.size))
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.legend()
        
        return fig


# Test GridWorld
gw = GridWorld(size=4, goal=(3, 3), obstacles=[(1, 1), (2, 2)], slip_prob=0.0)
print(f'GridWorld created: {len(gw.states)} states, {len(gw.actions)} actions')
print(f'States: {gw.states[:5]}...')
print(f'\nTest transition: state=(0,0), action=1 (down)')
next_s, reward = gw.next_state((0, 0), 1)
print(f'  Next state: {next_s}, Reward: {reward}')

<a name="inplace"></a>
## In-Place vs. Synchronous Updates

### The Difference

**Synchronous Update**:
```python
V_new = V.copy()
for s in states:
    V_new[s] = compute_value(s)  # Uses old V[]
V = V_new
```

**In-Place Update**:
```python
for s in states:
    V[s] = compute_value(s)  # Uses newest available V[]
```

In-place converges **faster** because we use updated values immediately.

### Mathematical Impact

Both converge to the same value V*, but in-place typically needs fewer iterations (20-30% faster empirically).

In [ ]:
def value_iteration_synchronous(gw, gamma=0.99, epsilon=1e-6, max_iter=1000):
    """Synchronous value iteration (copy V each iteration)."""
    V = {s: 0.0 for s in gw.states}
    history = []
    
    for iteration in range(max_iter):
        V_new = {}
        delta = 0
        
        for s in gw.states:
            max_q = -np.inf
            for a in gw.actions:
                trans, rewards = gw.get_transitions(s, a)
                q = 0.0
                for next_s, prob in trans.items():
                    r = rewards.get(next_s, 0.0)
                    q += prob * (r + gamma * V[next_s])
                max_q = max(max_q, q)
            
            V_new[s] = max_q
            delta = max(delta, abs(V[s] - V_new[s]))
        
        V = V_new
        history.append(delta)
        
        if delta < epsilon:
            return V, iteration + 1, history
    
    return V, max_iter, history


def value_iteration_inplace(gw, gamma=0.99, epsilon=1e-6, max_iter=1000):
    """In-place value iteration (update V directly)."""
    V = {s: 0.0 for s in gw.states}
    history = []
    
    for iteration in range(max_iter):
        delta = 0
        
        for s in gw.states:
            v_old = V[s]
            max_q = -np.inf
            
            for a in gw.actions:
                trans, rewards = gw.get_transitions(s, a)
                q = 0.0
                for next_s, prob in trans.items():
                    r = rewards.get(next_s, 0.0)
                    q += prob * (r + gamma * V[next_s])  # Uses newest V[]
                max_q = max(max_q, q)
            
            V[s] = max_q
            delta = max(delta, abs(v_old - V[s]))
        
        history.append(delta)
        
        if delta < epsilon:
            return V, iteration + 1, history
    
    return V, max_iter, history


# Compare on GridWorld
print('Comparing In-Place vs. Synchronous Updates\n' + '='*50)

start = time.time()
V_sync, iters_sync, hist_sync = value_iteration_synchronous(gw)
time_sync = time.time() - start

start = time.time()
V_inplace, iters_inplace, hist_inplace = value_iteration_inplace(gw)
time_inplace = time.time() - start

print(f'Synchronous: {iters_sync} iterations, {time_sync:.4f}s')
print(f'In-Place:    {iters_inplace} iterations, {time_inplace:.4f}s')
print(f'\nSpeedup: {time_sync / time_inplace:.2f}x (in-place faster)')
print(f'Difference in V(0,0): {abs(V_sync[(0,0)] - V_inplace[(0,0)]):.2e}')

<a name="async"></a>
## Asynchronous Dynamic Programming

### Algorithm

Update values in arbitrary order (not all states each iteration):

```
loop:
  Pick some state s (any order, any frequency)
  V(s) ← max_a sum_{s'} P(s'|s,a)[r + γV(s')]
  until convergence (if each state updated infinitely often)
```

### Why Asynchronous?

1. No need to visit all states each sweep
2. Can focus on important states first
3. Enables distributed/parallel implementations
4. Converges to same V* (guaranteed if each state visited ∞ times)

In [ ]:
def value_iteration_async(gw, gamma=0.99, epsilon=1e-6, max_updates=10000, update_order='cyclic'):
    """Asynchronous value iteration.
    
    Args:
        update_order: 'cyclic' (in order), 'random' (random pick), or a callable
    """
    V = {s: 0.0 for s in gw.states}
    n_states = len(gw.states)
    history = []
    update_counts = defaultdict(int)
    
    for update_num in range(max_updates):
        # Pick which state to update
        if update_order == 'cyclic':
            s = gw.states[update_num % n_states]
        elif update_order == 'random':
            s = gw.states[np.random.randint(n_states)]
        else:
            s = update_order(V)  # Callable: pick based on V
        
        v_old = V[s]
        max_q = -np.inf
        
        for a in gw.actions:
            trans, rewards = gw.get_transitions(s, a)
            q = 0.0
            for next_s, prob in trans.items():
                r = rewards.get(next_s, 0.0)
                q += prob * (r + gamma * V[next_s])
            max_q = max(max_q, q)
        
        V[s] = max_q
        update_counts[s] += 1
        
        delta = abs(v_old - V[s])
        if (update_num + 1) % n_states == 0:  # Log every sweep
            max_delta = max(delta for s in gw.states for delta in [abs(v_old - V[s])])
            history.append(max_delta)
            
            # Check if all states updated enough
            min_updates = min(update_counts.values())
            if min_updates > 0 and delta < epsilon:
                return V, update_num + 1, history, update_counts
    
    return V, max_updates, history, update_counts


# Test asynchronous DP
print('Asynchronous Value Iteration\n' + '='*50)

V_async_cyclic, updates_cyclic, hist_cyclic, counts_cyclic = value_iteration_async(gw, update_order='cyclic')
V_async_random, updates_random, hist_random, counts_random = value_iteration_async(gw, update_order='random')

print(f'Cyclic order:  {updates_cyclic} updates ({updates_cyclic//len(gw.states)} sweeps)')
print(f'Random order:  {updates_random} updates ({updates_random//len(gw.states)} sweeps)')
print(f'\nUpdate frequency (cyclic):')
for s, count in sorted(counts_cyclic.items())[:5]:
    print(f'  {s}: {count} updates')

<a name="prioritized"></a>
## Prioritized Sweeping

### Key Idea

Update states with **large Bellman residuals** first:

$$\text{Residual}(s) = |V(s) - \max_a Q(s, a)|$$

States with large residuals are furthest from their optimal value, so updating them first accelerates convergence.

### Algorithm

```
Initialize priority queue with all states (priority = 0)

loop:
  s ← pop state with highest priority
  if residual(s) < epsilon:
    break
  Update V(s)
  For each predecessor s' of s (states that can reach s):
    Update priority[s'] based on new residual
```

### Speedup

Prioritized sweeping often converges 2-5x faster than uniform updates because we focus on impactful states.

In [ ]:
import heapq

def value_iteration_prioritized(gw, gamma=0.99, epsilon=1e-6, max_updates=10000):
    """Prioritized sweeping (Bellman residual priority)."""
    V = {s: 0.0 for s in gw.states}
    
    # Build predecessor map: for each state, which states can reach it
    predecessors = defaultdict(set)
    for s in gw.states:
        for a in gw.actions:
            trans, _ = gw.get_transitions(s, a)
            for next_s in trans:
                predecessors[next_s].add((s, a))
    
    # Priority queue: (negative_residual, state)
    pq = []
    residuals = {}
    
    # Initialize with all states
    for s in gw.states:
        max_q = -np.inf
        for a in gw.actions:
            trans, rewards = gw.get_transitions(s, a)
            q = sum(prob * (rewards.get(next_s, 0.0) + gamma * V[next_s])
                   for next_s, prob in trans.items())
            max_q = max(max_q, q)
        residuals[s] = abs(V[s] - max_q)
        heapq.heappush(pq, (-residuals[s], s))
    
    history = []
    updates = 0
    
    while pq and updates < max_updates:
        neg_residual, s = heapq.heappop(pq)
        current_residual = residuals[s]
        
        # Skip if residual changed (stale priority)
        if -neg_residual != current_residual:
            continue
        
        if current_residual < epsilon:
            return V, updates, history, residuals
        
        # Update V(s)
        v_old = V[s]
        max_q = -np.inf
        for a in gw.actions:
            trans, rewards = gw.get_transitions(s, a)
            q = sum(prob * (rewards.get(next_s, 0.0) + gamma * V[next_s])
                   for next_s, prob in trans.items())
            max_q = max(max_q, q)
        
        V[s] = max_q
        updates += 1
        
        # Update priorities of predecessors
        for pred_s, _ in predecessors[s]:
            max_q_pred = -np.inf
            for a in gw.actions:
                trans, rewards = gw.get_transitions(pred_s, a)
                q = sum(prob * (rewards.get(next_s, 0.0) + gamma * V[next_s])
                       for next_s, prob in trans.items())
                max_q_pred = max(max_q_pred, q)
            
            new_residual = abs(V[pred_s] - max_q_pred)
            if new_residual > residuals.get(pred_s, 0):
                residuals[pred_s] = new_residual
                heapq.heappush(pq, (-new_residual, pred_s))
        
        if updates % len(gw.states) == 0:
            history.append(current_residual)
    
    return V, updates, history, residuals


# Compare prioritized vs uniform
print('Prioritized Sweeping vs. Uniform\n' + '='*50)

start = time.time()
V_prio, updates_prio, hist_prio, resid_prio = value_iteration_prioritized(gw)
time_prio = time.time() - start

print(f'Prioritized: {updates_prio} updates, {time_prio:.4f}s')
print(f'Uniform:     {iters_inplace} full iterations, {time_inplace:.4f}s')
print(f'\nSpeedup: {updates_prio / (iters_inplace * len(gw.states)):.2f}x (updates to convergence)')
print(f'\nExample residuals: {max(resid_prio.values()):.2e} → {min(resid_prio.values()):.2e}')

<a name="benchmarking"></a>
## Benchmarking & Comparison

### Experimental Setup

Test on GridWorlds of increasing size and compare:
- Iterations to convergence
- Wall-clock time
- Memory usage
- Effect of discount factor γ

In [ ]:
# Benchmark across grid sizes and algorithms
sizes = [4, 8, 12, 16]
results = {'size': [], 'sync': [], 'inplace': [], 'async_cyclic': [], 'prioritized': []}

print('Benchmarking across GridWorld sizes...')
for size in sizes:
    gw_test = GridWorld(size=size, goal=(size-1, size-1))
    print(f'\nSize {size}×{size} ({len(gw_test.states)} states):')
    
    # Synchronous
    start = time.time()
    _, iters, _ = value_iteration_synchronous(gw_test)
    t = time.time() - start
    print(f'  Synchronous: {iters} iterations, {t*1000:.1f}ms')
    results['sync'].append(t)
    
    # In-place
    start = time.time()
    _, iters, _ = value_iteration_inplace(gw_test)
    t = time.time() - start
    print(f'  In-place:    {iters} iterations, {t*1000:.1f}ms')
    results['inplace'].append(t)
    
    # Asynchronous cyclic
    start = time.time()
    _, updates, _, _ = value_iteration_async(gw_test, update_order='cyclic', max_updates=100000)
    t = time.time() - start
    print(f'  Async cyclic: {updates} updates ({updates//len(gw_test.states)} sweeps), {t*1000:.1f}ms')
    results['async_cyclic'].append(t)
    
    # Prioritized
    start = time.time()
    _, updates, _, _ = value_iteration_prioritized(gw_test)
    t = time.time() - start
    print(f'  Prioritized: {updates} updates, {t*1000:.1f}ms')
    results['prioritized'].append(t)
    
    results['size'].append(size)

# Plot results
fig, ax = plt.subplots(figsize=(10, 6))
for method in ['sync', 'inplace', 'async_cyclic', 'prioritized']:
    ax.plot(results['size'], results[method], 'o-', linewidth=2, markersize=8, label=method.replace('_', ' ').title())

ax.set_xlabel('GridWorld Size', fontsize=11)
ax.set_ylabel('Time (seconds)', fontsize=11)
ax.set_title('DP Algorithm Performance Comparison', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

<a name="debugging"></a>
## Debugging Convergence Issues

### Common Problems & Solutions

In [ ]:
debugging_guide = """
CONVERGENCE DEBUGGING CHECKLIST
================================

1. DIVERGENCE (Values increasing without bound)
   ✓ Check discount factor γ < 1
   ✓ Check rewards are reasonable (not ±inf)
   ✓ Verify transitions sum to 1.0 for all (s,a)
   ✓ Check for cycles with positive rewards

2. VERY SLOW CONVERGENCE
   ✓ If γ close to 1 (e.g., 0.999), convergence is inherently slow
   ✓ Reduce epsilon tolerance only after debugging
   ✓ Try prioritized sweeping or asynchronous updates
   ✓ Check if many states have similar Q-values (exploration vs exploitation)

3. WRONG VALUES (Converges to incorrect solution)
   ✓ Print some V(s) values and verify by hand
   ✓ Check if rewards are being applied correctly
   ✓ Verify terminal states have V=0 (or correct boundary)
   ✓ Check transition probabilities (should sum to 1.0)

4. POLICY NOT MAKING SENSE
   ✓ Policy extracted after convergence? (Not mid-iteration)
   ✓ Check for ties in argmax (multiple equally-good actions)
   ✓ Verify you're looking at greedy policy w.r.t. final V
   ✓ Check if goal state is actually reachable

5. NUMERICAL ISSUES (NaN, Inf)
   ✓ Use float64, not float32
   ✓ Check for invalid probability values (< 0 or > 1)
   ✓ Check for division by zero (prob = 0 in denominator)
   ✓ Log all V values periodically and check for anomalies

6. MEMORY ISSUES (Large problems)
   ✓ Use sparse representation for transitions
   ✓ Store only non-zero transitions
   ✓ Use function approximation instead of tabular
   ✓ Process in batches for very large problems
"""

print(debugging_guide)

<a name="gpi"></a>
## Advanced: Generalized Policy Iteration

### Hybrid Approaches

We can create hybrid algorithms by varying the evaluation and improvement steps:
- Full policy evaluation + greedy improvement = Policy Iteration
- One value iteration sweep + greedy improvement = One-step lookahead
- Asynchronous value updates + adaptive improvements = Prioritized methods

All converge to π* with appropriate conditions.

In [ ]:
def generalized_policy_iteration(gw, gamma=0.99, epsilon=1e-6, eval_steps=1, max_iters=100):
    """GPI with variable evaluation depth."""
    V = {s: 0.0 for s in gw.states}
    policy = {s: 0 for s in gw.states}
    
    for pi_iter in range(max_iters):
        # Policy Evaluation (eval_steps iterations)
        for eval_iter in range(eval_steps):
            delta = 0
            for s in gw.states:
                v_old = V[s]
                a = policy[s]
                trans, rewards = gw.get_transitions(s, a)
                V[s] = sum(prob * (rewards.get(next_s, 0.0) + gamma * V[next_s])
                          for next_s, prob in trans.items())
                delta = max(delta, abs(v_old - V[s]))
            
            if delta < epsilon:
                break
        
        # Policy Improvement
        policy_stable = True
        for s in gw.states:
            old_a = policy[s]
            q_values = []
            
            for a in gw.actions:
                trans, rewards = gw.get_transitions(s, a)
                q = sum(prob * (rewards.get(next_s, 0.0) + gamma * V[next_s])
                       for next_s, prob in trans.items())
                q_values.append(q)
            
            policy[s] = np.argmax(q_values)
            if policy[s] != old_a:
                policy_stable = False
        
        if policy_stable:
            return V, policy, pi_iter + 1
    
    return V, policy, max_iters


# Test GPI with different eval_steps
print('GPI with Variable Evaluation Depth\n' + '='*50)

for eval_steps in [1, 10, 100, 1000]:
    start = time.time()
    V, pi, iters = generalized_policy_iteration(gw, eval_steps=eval_steps)
    t = time.time() - start
    print(f'eval_steps={eval_steps:4d}: {iters} policy iterations, {t*1000:.1f}ms')

<a name="applications"></a>
## Real-World Applications

### When to Use DP in Practice

**Good fits**:
- Robot navigation (known map)
- Game planning (perfect information)
- Resource allocation (deterministic rewards)
- Scheduling (known durations)

**Challenges**:
- State space explosion (10⁸+ states)
- Unknown dynamics (must learn)
- Real-time constraints (can't wait for convergence)

### Scaling to Large Problems

1. **Function Approximation**: Replace V(s) table with V_θ(s) neural network
2. **Sampling**: Learn from experience instead of exact transitions
3. **Hierarchical DP**: Decompose into subproblems
4. **Distributed DP**: Parallelize over multiple machines

In [ ]:
# Final summary and recommendations
print('ALGORITHM SELECTION GUIDE')
print('='*60)
print()
print('Problem Size      Best Algorithm         Reason')
print('-'*60)
print('Small (< 100)     Policy Iteration       Exact convergence')
print('Medium (100-1k)   In-place Value Iter.   Simple & efficient')
print('Large (1k-10k)    Prioritized Sweeping   Focus on important states')
print('Very Large (>10k) Function Approx.       Tabular infeasible')
print()
print('Implementation Tips:')
print('  • Use in-place updates (faster convergence)')
print('  • Prioritize high-residual states')
print('  • Use float64 for precision')
print('  • Log convergence history for debugging')
print('  • Test policies by simulating episodes')